In [ ]:
# %% [markdown]
# # 2. BOE Subsidies & "Chiringuito" Detector
# # Source: BOE Open Data / BDNS
# # Output: subsidies_chiringuitos.json

import requests
import pandas as pd
import json
from datetime import datetime
from pydantic import BaseModel, Field
from typing import Literal

# --- Schema ---
class SubsidyRecord(BaseModel):
    id: str
    beneficiary: str
    amount_eur: float
    granting_body: str
    purpose: str
    date_granted: str
    source_url: str
    chiringuito_confidence: Literal["HIGH", "MEDIUM", "LOW", "NONE"]
    keywords_matched: list[str]

# --- Config ---
# Using BOE Open Data API endpoint for notifications (example)
BOE_API_URL = "https://www.boe.es/datosabiertos/api/api.php" 
# Note: For MVP, we might simulate data if API requires complex auth. 
# Below is a logic structure for when you have a CSV export from BDNS.
OUTPUT_FILE = "data/processed/subsidies_chiringuitos.json"

# --- Keywords ---
KEYWORDS_HIGH = ["concesión dominio público", "chiringuito", "barra de playa", "ocupación dominio público"]
KEYWORDS_MEDIUM = ["asociación sin ánimo de lucro", "subvención hostelería", "convenio colaboración"]
POLITICAL_FLAGS = ["fundación", "partido", "agrupación"]

def score_confidence(purpose: str, beneficiary: str, amount: float):
    text = f"{purpose} {beneficiary}".lower()
    matched = []
    
    for kw in KEYWORDS_HIGH:
        if kw in text: matched.append(kw)
    if matched and amount > 10000:
        return "HIGH", matched
        
    for kw in KEYWORDS_MEDIUM:
        if kw in text: matched.append(kw)
    if matched:
        # Check political connection hint
        if any(p in beneficiary.lower() for p in POLITICAL_FLAGS):
            return "HIGH", matched
        return "MEDIUM", matched
        
    return "NONE", matched

# --- Mock Fetch (Replace with actual CSV load if API fails) ---
def fetch_subsidies():
    print("📡 Fetching Subsidies (Simulated for MVP)...")
    # In production: requests.get(BOE_API_URL...) or pd.read_csv('bdns_export.csv')
    # Returning mock data to ensure app structure works immediately
    return [
        {
            "id": "BDNS-2023-001",
            "beneficiary": "Asociación Playa Limpia SL",
            "amount_eur": 45000.00,
            "granting_body": "Ayuntamiento de Marbella",
            "purpose": "Concesión dominio público marítimo-terrestre para barra de playa",
            "date_granted": "2023-06-15",
            "source_url": "https://www.boe.es"
        },
        {
            "id": "BDNS-2023-002",
            "beneficiary": "Fundación Cultura Ciudadana",
            "amount_eur": 120000.00,
            "granting_body": "Ministerio de Cultura",
            "purpose": "Subvención actividades culturales",
            "date_granted": "2023-07-10",
            "source_url": "https://www.boe.es"
        }
    ]

# --- Process ---
def process_subsidies(raw_data):
    records = []
    for item in raw_data:
        confidence, keywords = score_confidence(item['purpose'], item['beneficiary'], item['amount_eur'])
        
        # Only keep MEDIUM or HIGH for the dashboard focus
        if confidence in ["HIGH", "MEDIUM"]:
            records.append(SubsidyRecord(
                id=item['id'],
                beneficiary=item['beneficiary'],
                amount_eur=item['amount_eur'],
                granting_body=item['granting_body'],
                purpose=item['purpose'],
                date_granted=item['date_granted'],
                source_url=item['source_url'],
                chiringuito_confidence=confidence,
                keywords_matched=keywords
            ))
    return records

# --- Export ---
def export_data(records):
    import os
    os.makedirs("data/processed", exist_ok=True)
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            "meta": {
                "source": "BOE/BDNS",
                "generated_at": datetime.now().isoformat(),
                "filter": "Confidence HIGH/MEDIUM only"
            },
            "data": [r.model_dump() for r in records]
        }, f, indent=2, ensure_ascii=False)
    print(f"✅ Exported {len(records)} flagged subsidies to {OUTPUT_FILE}")

# --- Run ---
if __name__ == "__main__":
    raw = fetch_subsidies()
    clean = process_subsidies(raw)
    export_data(clean)